# **Section 2: Revision - Visualisation, functions, and grouping tables - Part 1**

Every cell here runs on its own, the data is built in the notebook rather than
loaded from a file, so you can change a number and re-run to see what happens.
That is the point: read the output, then break something deliberately.


#### **Helpful Resource:**
- [Python Reference](https://ulwazi.wits.ac.za/courses/89081/pages/detailed-python-reference-sheet-python-cheat-sheet-2?module_item_id=1200407)

**Recommended Readings:** 
- **Chapters 7 to 8.4 of the textbook**

**Where this is used.** This is the most directly useful section for **Project 1**.
`join`, `group`, `pivot` and `select` are the whole of the pipeline Project 1
builds in its first four sections, and this is the only place `join` and `pivot`
are taught.


In [ ]:
# Run this cell first -- the install takes about a minute in the browser
%pip install -q datascience ipywidgets

# pyodide_http is only present in JupyterLite -- ignored elsewhere
try:
    import pyodide_http
    pyodide_http.patch_all()
except ImportError:
    pass

from datascience import *
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
warnings.simplefilter('ignore', FutureWarning)

---

## **Contents**

1. [Choosing a plot](#1)
2. [Line and scatter plots](#2)
3. [Bar charts](#3)
4. [Histograms and the area principle](#4)
5. [Writing functions](#5)
6. [`apply`](#6)
7. [`group`](#7)
8. [Cross-classifying: `group` on two columns, and `pivot`](#8)
9. [`join`](#9)
10. [**>>Quick questions<<**](#10)
11. [**>>Self-check<<**](#11)
12. [Quick reference](#12)


---

<a id='1'></a>
## **1. Choosing a plot**

The plot you need depends on how many variables you have and whether each one is
**numerical** or **categorical**.

| What you have | Plot | Method |
|---|---|---|
| One categorical variable | Bar chart | `.barh(labels)` |
| One numerical variable | Histogram | `.hist(column, bins=...)` |
| Two numerical, x is ordered | Line | `.plot(x, y)` |
| Two numerical, x is not ordered | Scatter | `.scatter(x, y)` |
| One categorical, one numerical | Bar chart | `.barh(labels, values)` |

A column of numbers is not automatically numerical. Postal codes, years used as
labels, and ID numbers are all categorical, averaging them would be meaningless.


In [ ]:
# A small table we will reuse. Age is numerical, Studio is categorical.
movies = Table().with_columns(
    'Title',   make_array('Star Wars', 'Avatar', 'Titanic', 'Jaws', 'ET', 'Jurassic Park'),
    'Studio',  make_array('Fox', 'Fox', 'Paramount', 'Universal', 'Universal', 'Universal'),
    'Year',    make_array(1977, 2009, 1997, 1975, 1982, 1993),
    'Millions', make-array(3068, 890, 1270, 1170, 1330, 1140))
movies

---

<a id='2'></a>
## **2. Line and scatter plots**

Both take two numerical columns. The difference is whether the x values have a
natural order.

`.plot(x, y)` joins the points with a line, which says "as x increases, y does
this". Only use it when x is ordered, time, age, position.

`.scatter(x, y)` draws unconnected points. Use it when x has no order.


The table `population` below contains Population by age, Age is ordered

In [ ]:

population = Table().with-columns(
    'Age',        np.arange(0, 91, 10),
    'Millions',   make_array(3.9, 4.2, 4.4, 4.5, 4.1, 4.3, 3.6, 2.4, 1.3, 0.4)
    )

Population by age,  AGE is ordered, so a line makes sense

In [ ]:
population.plot('Age', 'Millions')

The same data as a scatter is correct, but the trend is harder to follow

In [ ]:
population.scatter('Age', 'Millions')

Now the other way round. In the table `actors`, Actors have no natural order, so joining them with a
line produces a mess.


In [ ]:
actors = Table().with_columns(
    'Actor',      make_array('Harrison Ford', 'Samuel Jackson', 'Morgan Freeman',
                             'Tom Hanks', 'Robert Downey Jr', 'Eddie Murphy'),
    'Movies',     make_array(41, 69, 61, 44, 53, 38),
    'Total Gross', make_array(4871, 4772, 4468, 4340, 3947, 3810)
    )

**WRONG:** Movies is not an ordered sequence, so the line is meaningless

In [ ]:
actors.plot('Movies', 'Total Gross')

**RIGHT:** the relationship is now readable

In [ ]:
actors.scatter('Movies', 'Total Gross')

> **Investigate anomalies.** A single point far from the rest is usually worth
> chasing down. In the real `actors.csv`, one actor has a very high average per
> movie, because they were in only a handful of enormously successful films.
> `.where(column, are.above(value))` is how you find out who.


---

<a id='3'></a>
## **3. Bar charts**

`.barh(labels)` on a table produced by `.group()` shows how many rows fall into
each category.

`.barh(labels, values)` shows one numerical value per category.

Sorting first almost always makes the chart easier to read.


**Example:** How many of these films did each studio make?

In [ ]:
movies.group('Studio')

In [ ]:
movies.group('Studio').barh('Studio')

- Sorted data is much easier to compare

In [ ]:
movies.group('Studio').sort('count', descending=True).barh('Studio')

- One bar per film, using a numerical column

In [ ]:
movies.sort('Millions', descending=True).barh('Title', 'Millions')

> **Titles matter.** A plot without a title makes the reader work out what they
> are looking at. Add one with `plots.title(...)` immediately after the plotting
> call, and say what the chart *shows*, not what it *is*, "Box office declines
> sharply after the top three" beats "Bar chart of millions".


---

<a id='4'></a>
## **4. Histograms and the area principle**

A histogram shows the distribution of **one numerical variable**. Values are
grouped into **bins**, and each bar covers a range rather than a single value.

```
tbl.hist(column, bins = np.arange(start, stop, step), unit = 'Year')
```

Bins can also be uneven, using `make_array`.


**Question:** How old are these films, in 2025?

In [ ]:
ages = 2025 - movies.column('Year')
movies = movies.with_columns('Age', ages)
movies.select('Title', 'Age')

- Even bins: every bar is 10 years wide

In [ ]:
movies.hist('Age', bins = np.arange(0, 61, 10), unit = 'Year')

- Uneven bins: narrow where the data is dense, wide where it is sparse

In [ ]:
my_bins = make_array(0, 5, 10, 15, 25, 40, 65)
movies.hist('Age', bins = my_bins, unit = 'Year')

### **The area principle**

**The y-axis is not a count.** It is *percent per unit*, density. What tells you
how much data is in a bin is the **area** of the bar: height × width.

This is why a wide, short bar can contain more data than a narrow, tall one.

$$\text{height} = \frac{\text{percent in bin}}{\text{width of bin}}
\qquad\qquad
\text{percent in bin} = \text{height} \times \text{width}$$

`.bin()` gives you the counts, so you can check this yourself.


In [ ]:
binned_data = movies.bin('Age', bins = my_bins)
binned_data

- Convert counts to percentages of all rows

In [ ]:
binned_data = binned_data.with_columns(
    'Percent', 100 * binned_data.column('Age count') / movies.num_rows)
binned_data

- The last row of `.bin()` is the right edge of the final bin, with count `0`.
- Drop it, then `np.diff` gives the width of each remaining bin.

In [ ]:
height_table = binned_data.take(np.arange(binned_data.num_rows - 1))
bin_widths = np.diff(binned_data.column('bin'))

height_table = height_table.with_columns('Width', bin_widths)
height_table = height_table.with_columns(
    'Height', height_table.column('Percent') / height_table.column('Width'))
height_table

Compare the `Height` column above with the bars in the histogram, they match.

> **Common mistake.** Reading a histogram as though the tallest bar has the most
> data. With uneven bins that is often false. Multiply height by width before
> comparing bins.


---

<a id='5'></a>
## **5. Writing functions**

```
def name(argument):
    """Documentation: what this does."""
    body
    return value
```

Four things to get right:

- **`def`** starts the definition; the line ends with a colon
- the **body is indented**, Python uses indentation, not brackets
- **`return`** hands a value back to whoever called the function
- the **docstring** in triple quotes says what it does


In [ ]:
def triple(x):
    """Triples the input."""
    tripled_x = x * 3
    return tripled_x

triple(3)

- The argument can be a name, an expression, an array, even a string

In [ ]:
num = 4
print(triple(num))
print(triple(num * 5))
print(triple(np.arange(4)))
print(triple('ha'))

### `return` is not `print`

`print` shows a value on the screen. `return` sends it back so it can be stored
or used. A function without `return` gives back `None`, silently.


In [ ]:
def double_print(x):
    print(x * 2)          # shows it, gives back nothing

def double_return(x):
    return x * 2          # hands the value back

a = double_print(5)
b = double_return(5)
print('from print:', a)   # None -- the value was displayed, not returned
print('from return:', b)

- Two arguments, and a string built from both

In [ ]:
def name_and_age(name, year):
    """Returns a sentence stating how old someone born in `year` is."""
    age = 2025 - year
    return name + ' is ' + str(age) + ' years old.'

name_and_age('Thandi', 1999)

---

<a id='6'></a>
## **6. `apply`**

`.apply(function, column)` runs a function on **every** value in a column and
returns an array of the results. It does not change the table.

Pass the function **without brackets**: you are giving `apply` the function
itself, not calling it.


In [ ]:
fam = Table().with_columns(
    'First Name', make_array('Thandi', 'Sipho', 'Naledi', 'Kagiso'),
    'Birth Year', make_array(1999, 1985, 2003, 1972))
fam

In [ ]:
def age(year):
    """How old someone born in `year` turns in 2025."""
    return 2025 - year

fam.apply(age, 'Birth Year')

- Two columns `->` a function that takes two arguments, in that order

In [ ]:
fam.apply(name_and_age, 'First Name', 'Birth Year')

- apply returns an array, so it can become a new column

In [ ]:
fam.with_columns('Statement', fam.apply(name_and_age, 'First Name', 'Birth Year'))

> **Common mistake.** Writing `fam.apply(age(), 'Birth Year')`. The brackets call
> the function immediately, before `apply` ever sees it. Leave them off.


---

<a id='7'></a>
## **7. `group`**

`.group(column)` counts how many rows share each value. The result always has a
column named `count`.

`.group(column, function)` instead applies a function to every other column,
one group at a time.


In [ ]:
cones = Table().with_columns(
    'Flavor', make_array('strawberry', 'chocolate', 'chocolate', 'strawberry',
                         'chocolate', 'bubblegum'),
    'Color',  make_array('pink', 'light brown', 'dark brown', 'pink',
                         'dark brown', 'pink'),
    'Price',  make_array(3.55, 4.75, 5.25, 5.25, 5.25, 4.75))
cones

**Question:** How many cones of each flavour?

In [ ]:
cones.group('Flavor')

- Average of every other column, per flavour.
- Color is text, so its average is blank -- that is expected.

In [ ]:
cones.group('Flavor', np.average)

- Drop the text column first for a cleaner result

In [ ]:
cones.drop('Color').group('Flavor', np.average)

> **Two things to watch.** `group` **renames** the aggregated column, `Price`
> becomes `Price average`. And it returns rows in **alphabetical** order of the
> grouping column, not the original order. Check before using `.item()` on the
> result.


---

<a id='8'></a>
## **8. Cross-classifying: `group` on two columns, and `pivot`**

Both answer the same question, how do two categorical variables combine?, but
they lay the answer out differently.

**`group([col1, col2])`** gives one row per combination. Long and narrow.

**`pivot(col1, col2)`** gives a grid: `col1` becomes the column headers, `col2`
the row labels. Wide, and easier to read across.


In [ ]:
cones.group(['Flavor', 'Color'])

In [ ]:
cones.pivot('Flavor', 'Color')

`pivot` also takes `values` and `collect`, to fill the grid with something other
than a count:

```
tbl.pivot(columns, rows, values = 'Price', collect = np.average)
```


In [ ]:
cones.pivot('Flavor', 'Color', values = 'Price', collect = np.average)

### **When to use which**

`pivot` is the better choice when you want to **compare two groups directly**, 
because they end up side by side as columns, so you can subtract them.


**Example:** The table below contains the tallest building of each material, in each city

In [ ]:
buildings = Table().with_columns(
    'city',     make_array('Chicago', 'Chicago', 'Chicago', 'New York',
                           'New York', 'New York', 'Durban', 'Durban'),
    'material', make_array('steel', 'concrete', 'steel', 'concrete',
                           'steel', 'concrete', 'concrete', 'steel'),
    'height',   make_array(442, 423, 346, 541, 319, 381, 152, 106))

- `group()`: one row per (city, material) pair

In [ ]:
buildings.group(['city', 'material'], max)

- `pivot()`: materials become columns, so they sit side by side

In [ ]:
tallest = buildings.pivot('material', 'city', values = 'height', collect = max)
tallest

- Because they are columns, subtracting them is a single step

In [ ]:
tallest = tallest.with_columns(
    'difference', abs(tallest.column('steel') - tallest.column('concrete')))
tallest.sort('difference', descending = True)

Doing that from the `group` version would mean pulling rows out one at a time.
When the next step is arithmetic *between* the groups, pivot first.


---

<a id='9'></a>
## **9. `join`**

`.join(own_column, other_table, other_column)` combines two tables by matching
values. The columns can have different names, which is what the third argument
is for.


- Create two tables `drinks` and `discounts`

In [ ]:
drinks = Table().with_columns(
    'Drink', make_array('Milk Tea', 'Espresso', 'Latte', 'Espresso'),
    'Cafe',  make_array('Asha', 'Strada', 'Strada', 'FSM'),
    'Price', make_array(5.5, 1.75, 3.25, 2.0))
drinks

In [ ]:
discounts = Table().with_columns(
    'Coupon',   make_array('10%', '25%', '5%'),
    'Location', make_array('Asha', 'Strada', 'Asha'))
discounts

- Match `drinks.Cafe` against `discounts.Location`

In [ ]:
drinks.join('Cafe', discounts, 'Location')

Look carefully at what happened:

- **FSM has vanished.** It appears in `drinks` but not in `discounts`, so it is
  dropped. This is an *inner* join, rows without a match on both sides disappear,
  with no warning.
- **Asha appears twice.** It has two coupons, so every Asha drink is paired with
  each of them.

Swapping which table you start from changes the column order, but the matching
is the same.


In [ ]:
discounts.join('Location', drinks, 'Cafe')

> **Common mistake.** Assuming `join` keeps everything, like a spreadsheet lookup
> that leaves blanks. It does not. Compare `num_rows` before and after, if rows
> went missing, that is why.


---
<a id='10'></a>
## **10. Quick questions**

Five of them. Set `my_answer` to a letter and run the cell.
A wrong answer gets a nudge so you can try again; a right one gets the reason.

These come before the written questions below on purpose: they are quicker,
and they check the things people most often get wrong.

In [ ]:
# Run this once. mcq.py must be in the same folder as this notebook.
from mcq import check_answer, show_answer

**M1.** After `sales.group('Region', np.average)`, what is the `Price` column called?

**a)** `Price`  
**b)** `average Price`  
**c)** `Price average`  
**d)** `Region average`  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w2_m1', my_answer)

**M2.** Which is the correct call?

**a)** `tbl.apply(double(), 'Price')`  
**b)** `tbl.apply(double, 'Price')`  
**c)** `tbl.apply('double', 'Price')`  
**d)** `double(tbl.apply('Price'))`  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w2_m2', my_answer)

**M3.** A histogram has a bin from 30 to 40 and one from 40 to 80. The second bar is half the height. Which bin holds more?

**a)** the first, twice as much  
**b)** the first, four times as much  
**c)** they hold the same  
**d)** the second, twice as much  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w2_m3', my_answer)

**M4.** You join two 90-row tables and get 71 rows. What happened?

**a)** 19 rows had no match in one of the tables and were dropped  
**b)** `join` removed duplicates  
**c)** the join column had missing values  
**d)** nothing, `join` always shrinks a table  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w2_m4', my_answer)

**M5.** A function ends with `print(answer)` instead of `return answer`. What does it give back?

**a)** `answer`  
**b)** the printed text as a string  
**c)** `None`  
**d)** an error  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w2_m5', my_answer)

---
<a id='11'></a>
## **11. Self-check**

Answer these without scrolling back. Reveal each answer only after you have committed to one.

**Q1.** You run `sales.group('Region', np.average)`. The original table had a `Price` column. What is that column called now, and why does `sales.column('Price')` fail afterwards?

<details>
<summary><strong>Answer</strong></summary>

It is called <code>Price average</code>. <code>group</code> renames every aggregated column by appending the name of the function, so the old label no longer exists and <code>column('Price')</code> raises a <code>KeyError</code>. Check <code>.labels</code> after any <code>group</code> with a function.

</details>

**Q2.** What is wrong with `tbl.apply(double(), 'Price')`, and what does the correct call look like?

<details>
<summary><strong>Answer</strong></summary>

The brackets call <code>double</code> immediately, before <code>apply</code> ever sees it, so Python raises an error about missing arguments. Pass the function itself: <code>tbl.apply(double, 'Price')</code>. <code>apply</code> does the calling, once per row.

</details>

**Q3.** Table A has 50 rows, table B has 50 rows, and you join them on an ID column. Can the result have more than 50 rows? Can it have fewer?

<details>
<summary><strong>Answer</strong></summary>

Fewer, yes: <code>join</code> keeps only rows whose key appears in <strong>both</strong> tables, so unmatched rows silently disappear. More is also possible if a key repeats in one table, since each match produces a row. Always compare <code>num_rows</code> before and after.

</details>

**Q4.** You have a table of cities and their populations, in no particular order. Why is `tbl.plot('City', 'Population')` the wrong choice?

<details>
<summary><strong>Answer</strong></summary>

<code>plot</code> draws a line, which claims the points are in a meaningful order and that the space between them means something. City names are categorical and unordered, so the line is meaningless. Use <code>barh</code> for a categorical variable, and keep <code>plot</code> for something ordered like time.

</details>

**Q5.** A histogram has bins of unequal width. Why can you not compare two bars by their height alone?

<details>
<summary><strong>Answer</strong></summary>

Because of the area principle: in a density histogram it is the <strong>area</strong> of a bar, not its height, that gives the proportion of data in that bin. A wide bin can hold more data than a taller narrow one. Multiply height by width to compare.

</details>

**Q6.** `group(['Region', 'Product'])` and `pivot('Region', 'Product')` count the same combinations. When would you reach for each?

<details>
<summary><strong>Answer</strong></summary>

<code>group</code> with a list gives a <strong>long</strong> table, one row per combination, which is what you want when the result feeds more code. <code>pivot</code> gives a <strong>wide</strong> grid, one row per value of one variable and one column per value of the other, which is easier for a person to read. Same information, different shape.

</details>

---

<a id='12'></a>
## **12. Quick reference**

### **Plotting**

| Call | Shows |
|---|---|
| `tbl.plot(x, y)` | Line, only when x is ordered |
| `tbl.scatter(x, y)` | Scatter, two numerical variables |
| `tbl.barh(labels)` | Bar chart of counts, after `.group()` |
| `tbl.barh(labels, values)` | Bar chart of one numerical column |
| `tbl.hist(col, bins=..., unit=...)` | Histogram of one numerical variable |
| `tbl.bin(col, bins=...)` | The counts behind a histogram, as a table |

### **Functions**

| Call | Does |
|---|---|
| `def f(x):` … `return y` | Defines a function |
| `tbl.apply(f, col)` | Runs `f` on every value, returns an array |
| `tbl.apply(f, col1, col2)` | Runs a two-argument `f` down both columns |

### **Grouping and combining**

| Call | Returns |
|---|---|
| `tbl.group(col)` | One row per value, with a `count` column |
| `tbl.group(col, fn)` | One row per value, `fn` applied to other columns |
| `tbl.group([c1, c2])` | One row per combination, long |
| `tbl.pivot(cols, rows)` | A grid of counts, wide |
| `tbl.pivot(cols, rows, values, collect)` | A grid of aggregated values |
| `tbl.join(own, other_tbl, theirs)` | Matched rows from both, unmatched dropped |

### **Things that catch people out**

| | |
|---|---|
| Histogram y-axis | Density, not count. Area = percent. |
| `group` column names | `Price` becomes `Price average` |
| `group` row order | Alphabetical, not original |
| `apply` | Pass `f`, not `f()` |
| A function with no `return` | Gives back `None` |
| `join` | Inner join, unmatched rows disappear |
| `.bin()` last row | Right edge of the final bin, count 0 |

---

### **Where this comes from in the textbook**

- [Chapter 7, Visualization](https://inferentialthinking.com/chapters/07/visualization/)
- [Chapter 7.1, Categorical distributions](https://inferentialthinking.com/chapters/07/1/visualizing-categorical-distributions/)
- [Chapter 7.2, Numerical distributions](https://inferentialthinking.com/chapters/07/2/visualizing-numerical-distributions/)
- [Chapter 8, Functions and Tables](https://inferentialthinking.com/chapters/08/functions-and-tables/)
- [Chapter 8.1, Applying a function to a column](https://inferentialthinking.com/chapters/08/1/applying-a-function-to-a-column/)
- [Chapter 8.2, Classifying by one variable](https://inferentialthinking.com/chapters/08/2/classifying-by-one-variable/)
- [Chapter 8.3, Cross-classifying](https://inferentialthinking.com/chapters/08/3/cross-classifying-by-more-than-one-variable/)
- [Chapter 8.4, Joining tables by columns](https://inferentialthinking.com/chapters/08/4/joining-tables-by-columns/)
